# 🚀 JN_Launch_Computation — Launch a TsunamiHySEA Computation

**HySEALab · Processing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

Interactive notebook to select a simulation folder and parfile, inspect the
domain, configure GPUs, launch the computation and **follow its progress in
real time** — all from JupyterLab, without opening a terminal.

> ⚠️ **IMPORTANT — where this notebook must run**
>
> Unlike the preprocessing notebooks, this one is **not** machine-independent:
> the Jupyter **kernel must run on the machine that hosts the GPU(s)** and the
> Tsunami-HySEA installation (binaries, MPI, CUDA). Typically you open
> JupyterLab on the compute server itself (or through a remote session whose
> kernel lives there). Running it on your laptop against a remote server will
> not work — the launch buttons execute `mpirun`/`TsunamiHySEA` locally.
>
> The first code cell runs an **environment check** that verifies every
> external dependency (✓/✗) before you go any further.

---

### Requirements

**Python packages** (in the kernel environment):

```bash
conda install -c conda-forge numpy matplotlib ipywidgets netcdf4 psutil
# optional, nicer file dialogs: pip install ipyfilechooser
```

- `psutil` — recommended; gives the **Stop** button a robust process-tree kill
  (without it a simpler `killpg` fallback is used)
- `ipyfilechooser` — optional; a built-in folder browser is used if absent

**On the compute machine** (paths configured in the first code cell —
**adapt them to your system**):

- Tsunami-HySEA binaries (`TsunamiHySEA`, `get_load_balancing`)
- An MPI installation (`mpirun`)
- HDF5 / NetCDF runtime libraries
- A conda environment for the GPU runtime (default name: `lab_gpu`)
- One or more CUDA-capable GPUs

**Input data:** a simulation folder containing a valid parameter file and the
bathymetry grids it references (see the preprocessing notebooks
[JN04](../preprocessing/JN04_Grid_from_GEBCO.ipynb),
[JN12](../preprocessing/JN12_Format_Conversion.ipynb) and
[JN13](../preprocessing/JN13_Parameter_File_Builder_and_Validator.ipynb)).

---

**Workflow:**

| Step | Action |
|------|--------|
| 1 | Select working directory |
| 2 | Select parfile + inspect/edit contents |
| 3 | Plot domain bathymetry (all nesting levels) |
| 4 | Configure GPUs → run load balancing |
| 5 | Launch TsunamiHySEA (`mpirun` inside `conda run`) |
| 6 | Monitor log in real time (progress bar + auto-refresh) |

In [ ]:
import subprocess, os, glob, time, threading, signal
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print('⚠  psutil not found — stop button will use os.killpg fallback')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from netCDF4 import Dataset

try:
    from ipyfilechooser import FileChooser
    HAS_FILECHOOSER = True
except ImportError:
    HAS_FILECHOOSER = False

# ═══ USER CONFIGURATION ═════════════════════════════════════════════════════
# Adapt every path below to YOUR compute machine. The startup check at the
# end of this cell verifies each item and reports ✓/✗.

# ── HySEA binary paths ────────────────────────────────────────────────────────
HYSEA_BASE  = os.path.expanduser('~/HySEALab/codes/Tsunami-HySEA_v4.4.1')
LB_EXE      = os.path.join(HYSEA_BASE, 'bin_lb', 'get_load_balancing')
HYSEA_EXE   = os.path.join(HYSEA_BASE, 'bin', 'TsunamiHySEA')
CONDA_ENV   = 'lab_gpu'

# ── System library paths ──────────────────────────────────────────────────────
SYS_HDF5_LIB   = '/share/local/hdf5/lib'
SYS_NETCDF_LIB = '/share/local/netcdf/lib'

# ── MPI configuration ─────────────────────────────────────────────────────────
MPIRUN_EXE    = '/usr/local/openmpi/bin/mpirun'
EXTRA_PATH_DIRS = ['/usr/local/openmpi/bin']

# ─────────────────────────────────────────────────────────────────────────────
def _make_env_preamble():
    """Return bash lines that set up LD_LIBRARY_PATH and PATH."""
    extra_path = ':'.join(EXTRA_PATH_DIRS)
    return (
        f'export LD_LIBRARY_PATH={SYS_HDF5_LIB}:{SYS_NETCDF_LIB}:$LD_LIBRARY_PATH\n'
        f'export PATH={extra_path}:$PATH\n'
    )

# ── Startup check ─────────────────────────────────────────────────────────────
# Validate every external dependency up front so a wrong path or a missing
# binary is caught here (✓/✗) instead of mid-demo inside a button callback.
import sys, shutil

def _check(ok, label, detail):
    print(f'{"✓" if ok else "✗"}  {label:<22}{detail}')
    return ok

print('✓  Imports OK\n')
print('Environment check:')
_failures = 0

# conda env — kernel may run inside it, or it may live alongside in envs/
_env_path = os.path.join(os.path.dirname(sys.prefix), CONDA_ENV)
_env_ok   = (os.path.basename(sys.prefix) == CONDA_ENV) or os.path.isdir(_env_path)
_failures += not _check(_env_ok, f"conda env '{CONDA_ENV}'",
                        _env_path if _env_ok else 'NOT FOUND')

# Tsunami-HySEA executables
_failures += not _check(os.path.isfile(LB_EXE), 'get_load_balancing',
                        LB_EXE if os.path.isfile(LB_EXE) else f'NOT FOUND at {LB_EXE}')
_failures += not _check(os.path.isfile(HYSEA_EXE), 'TsunamiHySEA',
                        HYSEA_EXE if os.path.isfile(HYSEA_EXE) else f'NOT FOUND at {HYSEA_EXE}')

# mpirun — accept the configured path or any mpirun on PATH
_mpi = MPIRUN_EXE if os.path.isfile(MPIRUN_EXE) else (shutil.which('mpirun') or '')
_failures += not _check(bool(_mpi), 'mpirun',
                        _mpi or f'NOT FOUND at {MPIRUN_EXE}')

# Runtime library directories
for _lib in (SYS_HDF5_LIB, SYS_NETCDF_LIB):
    _name = os.path.basename(os.path.dirname(_lib)) + ' lib'
    _failures += not _check(os.path.isdir(_lib), _name,
                            _lib if os.path.isdir(_lib) else f'NOT FOUND at {_lib}')

# Optional: directory browser backend (built-in navigator works without it)
_check(HAS_FILECHOOSER, 'ipyfilechooser',
       'available' if HAS_FILECHOOSER else 'optional — using built-in folder browser')

print()
if _failures:
    print(f'⚠  {_failures} required item(s) missing — fix the paths in this cell '
          'before running Steps 4–5 (load balancing / launch).')
else:
    print('✓  All required dependencies found — ready to run.')

---
## Step 1 — Select working directory

Choose the folder that contains the parfile and bathymetry files.

In [ ]:
# ── Shared state ──────────────────────────────────────────────────────────────
state = {
    'sim_dir': None,
    'parfile': None,
    'n_gpus': 1,
    'n_gpus_launch': 1,
    'log_file': None,
    'proc': None,
}

# ── Directory browser ─────────────────────────────────────────────────────────
# Interactive navigator: breadcrumb + subdirectory dropdown + select button.
# No external dependencies required beyond ipywidgets.

_NAV_ROOT = os.path.expanduser('~')   # starting root for navigation

def _subdirs(path):
    """Sorted list of subdirectory names in path (ignores hidden and unreadable)."""
    try:
        return sorted(
            d for d in os.listdir(path)
            if not d.startswith('.') and os.path.isdir(os.path.join(path, d))
        )
    except PermissionError:
        return []

# ── Widgets ───────────────────────────────────────────────────────────────────
breadcrumb_html = widgets.HTML(value='')
subdir_dropdown  = widgets.Dropdown(
    options=[],
    description='Enter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='55%'),
)
btn_enter   = widgets.Button(description='▶ Open', button_style='info',
                              layout=widgets.Layout(width='90px'))
btn_up      = widgets.Button(description='↑ Up',   button_style='',
                              layout=widgets.Layout(width='80px'))
btn_select  = widgets.Button(description='✓ Select this folder', button_style='success',
                              layout=widgets.Layout(width='200px'))
selected_html = widgets.HTML(value='<i style="color:#888">No folder selected yet.</i>')

_current_path = [_NAV_ROOT]   # mutable container so closures can update it

def _refresh_browser():
    path = _current_path[0]
    # Breadcrumb: each part is a clickable HTML span (uses JS postMessage trick
    # not available in all frontends, so we just show the path prettily)
    parts = []
    accumulated = '/'
    for part in path.replace('\\', '/').split('/'):
        if not part:
            continue
        accumulated = os.path.join(accumulated, part)
        parts.append(f'<span style="color:#2980b9;font-weight:bold">{part}</span>')
    crumb = ' <span style="color:#aaa">/</span> '.join(parts)
    breadcrumb_html.value = (
        f'<div style="font-family:monospace;font-size:13px;padding:4px 0">'
        f'📁 {crumb}</div>'
    )
    # Subdirectory list
    subs = _subdirs(path)
    if subs:
        subdir_dropdown.options = subs
        subdir_dropdown.value   = subs[0]
        subdir_dropdown.disabled = False
        btn_enter.disabled = False
    else:
        subdir_dropdown.options  = ['(no subdirectories)']
        subdir_dropdown.disabled = True
        btn_enter.disabled = True

    btn_up.disabled = (os.path.abspath(path) == os.path.abspath(_NAV_ROOT))

def _on_enter(_):
    chosen = subdir_dropdown.value
    if chosen and not chosen.startswith('('):
        _current_path[0] = os.path.join(_current_path[0], chosen)
        _refresh_browser()

def _on_up(_):
    parent = os.path.dirname(_current_path[0])
    if os.path.isdir(parent):
        _current_path[0] = parent
        _refresh_browser()

def _on_select(_):
    p = _current_path[0]
    state['sim_dir'] = p
    selected_html.value = (
        f'<b style="color:green">✓ Selected:</b> '
        f'<span style="font-family:monospace">{p}</span>'
    )

btn_enter.on_click(_on_enter)
btn_up.on_click(_on_up)
btn_select.on_click(_on_select)

# Initialise
_refresh_browser()

display(
    widgets.HTML('<b>Step 1 — Navigate to simulation directory</b>'),
    breadcrumb_html,
    widgets.HBox([subdir_dropdown, btn_enter, btn_up]),
    btn_select,
    selected_html,
)

---
## Step 2 — Select parfile and inspect contents

All `.txt` files in the selected directory are listed as options.

In [ ]:
out_parfile = widgets.Output()


# ─────────────────────────────────────────────────────────────────────────────
# Robust parfile reader
# ─────────────────────────────────────────────────────────────────────────────
# Why this helper exists:
#  · Default open() uses the platform encoding (cp1252 on Windows) which
#    fails on parfiles that contain UTF-8 comments or a BOM.
#  · We try UTF-8 (with BOM stripping) first, then fall back to latin-1
#    which never raises on bytes.
# ─────────────────────────────────────────────────────────────────────────────
def _read_parfile_text(path):
    """Read a parfile and return (text, encoding_used).

    Raises FileNotFoundError if path does not exist.
    Never raises UnicodeDecodeError: falls back to latin-1.
    """
    if not path or not os.path.isfile(path):
        raise FileNotFoundError(f'Parfile not found: {path}')
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            with open(path, encoding=enc) as fh:
                return fh.read(), enc
        except UnicodeDecodeError:
            continue
    # Should be unreachable — latin-1 always succeeds
    raise RuntimeError(f'Could not decode {path} with any tried encoding')


def _scan_parfiles(_=None):
    with out_parfile:
        clear_output()
        d = state['sim_dir']
        if not d or not os.path.isdir(d):
            print('⚠  Run Step 1 first and select a valid directory.')
            return
        txt_files = sorted(glob.glob(os.path.join(d, '*.txt')))
        if not txt_files:
            print(f'⚠  No .txt files found in {d}')
            return

        options = [os.path.basename(f) for f in txt_files]

        # ── Widgets ───────────────────────────────────────────────────────────
        dd = widgets.Dropdown(
            options=options, value=options[0],
            description='Parfile:',
            layout=widgets.Layout(width='55%'),
        )
        btn_load = widgets.Button(description='Load', button_style='info',
                                  icon='folder-open', layout=widgets.Layout(width='90px'))
        tab_label = widgets.HTML(
            value='<b style="color:#888">Select a parfile and click Load.</b>'
        )

        # ── Tabs: View | Edit ─────────────────────────────────────────────────
        # View tab
        out_view = widgets.Output(
            layout=widgets.Layout(
                border='1px solid #ddd', padding='6px',
                max_height='380px', overflow_y='auto', width='100%',
            )
        )

        # Edit tab
        editor = widgets.Textarea(
            value='',
            placeholder='Parfile content will appear here after loading.',
            layout=widgets.Layout(width='100%', height='360px'),
        )
        save_status = widgets.HTML(value='')
        btn_save    = widgets.Button(description='💾  Save parfile', button_style='warning',
                                     layout=widgets.Layout(width='180px'))
        btn_reload  = widgets.Button(description='↺  Reload from disk', button_style='',
                                     layout=widgets.Layout(width='180px'))
        edit_bar    = widgets.HBox([btn_save, btn_reload, save_status])

        out_edit = widgets.VBox([edit_bar, editor])

        tabs = widgets.Tab(children=[out_view, out_edit])
        tabs.set_title(0, '👁 View')
        tabs.set_title(1, '✏ Edit')

        # ── Load callback ─────────────────────────────────────────────────────
        def _load_file(path):
            # Read with the robust helper. Any failure here is shown in BOTH
            # tabs and in save_status so the user is never left guessing.
            try:
                text, enc = _read_parfile_text(path)
            except Exception as e:
                err_html = (
                    f'<b style="color:red">✗ Could not read parfile</b><br>'
                    f'<span style="font-family:monospace">{path}</span><br>'
                    f'<span style="color:#a00">{type(e).__name__}: {e}</span>'
                )
                with out_view:
                    clear_output()
                    display(HTML(err_html))
                editor.value = ''
                save_status.value = err_html
                return

            # -- View tab: syntax-highlighted HTML
            with out_view:
                clear_output()
                lines_html = []
                for ln in text.splitlines():
                    stripped = ln.strip()
                    # Escape HTML special chars
                    safe = ln.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
                    if stripped.startswith('#'):
                        lines_html.append(f'<span style="color:#888">{safe}</span>')
                    elif stripped.lower().startswith(('begin', 'end')):
                        lines_html.append(
                            f'<span style="color:#1a5276;font-weight:bold">{safe}</span>'
                        )
                    else:
                        parts = safe.split('#', 1)
                        body = parts[0]
                        cmt  = f'<span style="color:#888">#{parts[1]}</span>' if len(parts) > 1 else ''
                        lines_html.append(f'{body}{cmt}')
                enc_note = '' if enc == 'utf-8' else (
                    f'<div style="color:#b58105;font-size:11px;margin-bottom:4px">'
                    f'ℹ Decoded as <b>{enc}</b> '
                    f'(parfile was not strict UTF-8)</div>'
                )
                display(HTML(
                    enc_note +
                    '<pre style="font-size:12px;line-height:1.5;margin:0">'
                    + '<br>'.join(lines_html)
                    + '</pre>'
                ))

            # -- Edit tab: raw text in Textarea
            editor.value = text
            save_status.value = ''

        def _on_load(_b):
            chosen = os.path.join(d, dd.value)
            state['parfile'] = chosen
            tab_label.value = (
                f'<b>Loaded:</b> <span style="font-family:monospace">{chosen}</span>'
            )
            _load_file(chosen)

        def _on_save(_b):
            path = state.get('parfile')
            if not path:
                save_status.value = '<b style="color:red">No parfile selected.</b>'
                return
            try:
                # Always write UTF-8 — the standard for HySEA parfiles.
                with open(path, 'w', encoding='utf-8') as fh:
                    fh.write(editor.value)
                save_status.value = '<b style="color:green">✓ Saved</b>'
                # Refresh view tab
                _load_file(path)
            except Exception as e:
                save_status.value = (
                    f'<b style="color:red">✗ {type(e).__name__}: {e}</b>'
                )

        def _on_reload(_b):
            path = state.get('parfile')
            if path:
                _load_file(path)
                save_status.value = '<b style="color:#888">Reloaded from disk.</b>'

        btn_load.on_click(_on_load)
        btn_save.on_click(_on_save)
        btn_reload.on_click(_on_reload)

        # Auto-load first file
        state['parfile'] = os.path.join(d, options[0])
        _load_file(state['parfile'])
        tab_label.value = (
            f'<b>Loaded:</b> <span style="font-family:monospace">{state["parfile"]}</span>'
        )

        display(
            widgets.HBox([dd, btn_load]),
            tab_label,
            tabs,
        )
        print(f'Found {len(txt_files)} .txt file(s).')

btn_scan = widgets.Button(description='Scan directory', button_style='primary', icon='search')
btn_scan.on_click(_scan_parfiles)
display(btn_scan, out_parfile)

---
## Step 3 — Plot domain bathymetry

Reads the **first (coarsest) grid** listed in the parfile and plots the domain.  
The grid file must be a NetCDF `.grd` file with variables `x`, `y`, `z`.

In [ ]:
import re
import matplotlib.patches as mpatches
import matplotlib.cm as cm

out_map = widgets.Output()

# ── Grid file extensions recognised as bathymetry ─────────────────────────────
_BATHY_EXTS = ('.grd', '.nc', '.nc4', '.netcdf')

# ── Variable name candidates (tried in order) ─────────────────────────────────
_X_NAMES = ('x', 'lon', 'longitude', 'LON', 'LONGITUDE', 'X')
_Y_NAMES = ('y', 'lat', 'latitude',  'LAT', 'LATITUDE',  'Y')
_Z_NAMES = ('z', 'z_range', 'Band1', 'elevation', 'depth',
            'bathymetry', 'topo', 'ELEVATION', 'DEPTH', 'Z')


# ── Robust parfile text reader (shared with Step 2) ──────────────────────────
def _read_parfile_lines(parfile_path):
    """Return parfile lines decoded with UTF-8 (BOM-tolerant) and a
    latin-1 fallback. Empty list if the file does not exist."""
    if not parfile_path or not os.path.isfile(parfile_path):
        return []
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            with open(parfile_path, encoding=enc) as fh:
                return fh.read().splitlines(keepends=True)
        except UnicodeDecodeError:
            continue
    return []


# ── Helpers ───────────────────────────────────────────────────────────────────

def _hysea_level_from_name(basename):
    """_L0, _L1 … suffix in filename → int, else None."""
    m = re.search(r'_L(\d+)', basename, re.IGNORECASE)
    return int(m.group(1)) if m else None

def _hysea_level_from_comment(comment):
    """'Bathymetry file for level N' in inline comment → int, else None."""
    m = re.search(r'[Bb]athymetry[^#\n]*[Ll]evel\s+(\d+)', comment)
    return int(m.group(1)) if m else None

def _is_bathy(token):
    return any(token.lower().endswith(ext) for ext in _BATHY_EXTS)

def _is_int(s):
    try: int(s); return True
    except: return False

def _is_float_row(s):
    """Returns True for fault-parameter lines (≥5 numbers)."""
    parts = s.split()
    if len(parts) < 5:
        return False
    try: [float(p) for p in parts]; return True
    except: return False

# ── Structural parser ─────────────────────────────────────────────────────────

def _parse_levels_structural(parfile_path):
    """
    Parse the HySEA nested-grid parfile format to map each bathymetry
    basename to its nesting level.  Returns dict {basename: level}.

    Format (content lines after stripping blank lines and comments):
      [0] Title
      [1] L0 bathy file
      [2] Initialization type (int)
      [3] Kajiura filter (int)
      [4] Number of faults N (int)
      [5..4+N] Fault parameter rows (floats)
      [4+N+1] Okada window (int)
      [4+N+2] L0 NetCDF prefix
      [4+N+3] L0 variable save flags
      [4+N+4] Number of levels M (int)
      For each level 1..M-1:
        Refinement ratio (int)
        Number of submeshes K (int)
        For each submesh 1..K:
          Bathy file
          NC prefix
          Variable save flags
    """
    base_dir = os.path.dirname(parfile_path)
    raw_lines = _read_parfile_lines(parfile_path)
    cl = [l.strip().split('#')[0].strip() for l in raw_lines]
    cl = [l for l in cl if l]

    result = {}
    idx = [0]

    def nxt():
        if idx[0] >= len(cl): return None
        v = cl[idx[0]]; idx[0] += 1; return v

    # Title
    nxt()
    # L0 bathy
    l0 = nxt()
    if l0 and _is_bathy(l0):
        result[os.path.basename(l0)] = 0
    else:
        return result
    # Init type
    if not _is_int(nxt() or ''): return result
    # Kajiura
    if not _is_int(nxt() or ''): return result
    # Number of faults
    nf_str = nxt()
    if not _is_int(nf_str or ''): return result
    num_faults = int(nf_str)
    # Skip fault lines (floats) — skip exactly num_faults float-heavy rows
    skipped = 0
    while skipped < num_faults:
        tok = nxt()
        if tok is None: return result
        if _is_float_row(tok):
            skipped += 1
        # Non-float row while waiting → parsing error
        elif skipped < num_faults:
            return result
    # Okada window
    nxt()
    # L0 NC prefix
    nxt()
    # L0 var flags
    nxt()
    # Number of levels
    nl_str = nxt()
    if not _is_int(nl_str or ''): return result
    num_levels = int(nl_str)

    for lvl in range(1, num_levels):
        if not _is_int(nxt() or ''): return result   # refinement ratio
        ns_str = nxt()
        if not _is_int(ns_str or ''): return result  # num submeshes
        for _ in range(int(ns_str)):
            bathy = nxt()
            if bathy is None: return result
            if _is_bathy(bathy):
                result[os.path.basename(bathy)] = lvl
            else:
                return result   # unexpected
            nxt()   # NC prefix
            nxt()   # var flags

    return result

# ── Main grid-file parser ─────────────────────────────────────────────────────

def _parse_grid_files(parfile_path):
    """
    Return list of (basename, abs_path, level_int_or_None).

    Level detection priority:
      1. Inline comment on same line  → '# Bathymetry file for level N'
      2. Filename suffix              → _L0, _L1 …
      3. Structural parsing of parfile (HySEA nested-grid format)
    """
    if not parfile_path or not os.path.isfile(parfile_path):
        return []

    base_dir = os.path.dirname(parfile_path)
    raw = []   # (basename, abs_path, lvl_comment, lvl_name)

    for line in _read_parfile_lines(parfile_path):
        parts = line.strip().split('#', 1)
        token = parts[0].strip()
        comment = parts[1].strip() if len(parts) > 1 else ''
        if not token or not _is_bathy(token):
            continue
        p = token if os.path.isabs(token) else os.path.join(base_dir, token)
        name = os.path.basename(token)
        raw.append((name, p,
                    _hysea_level_from_comment(comment),
                    _hysea_level_from_name(name)))

    if not raw:
        return []

    # Merge strategies 1 and 2
    result = [(n, p, lc if lc is not None else ln) for n, p, lc, ln in raw]

    # Strategy 3: structural fallback for any remaining None levels
    if any(lvl is None for _, _, lvl in result):
        try:
            struct = _parse_levels_structural(parfile_path)
            result = [(n, p, struct.get(n, lvl)) for n, p, lvl in result]
        except Exception:
            pass

    return result

def _effective_level(lvl):
    """None → 0  (unlabelled grids treated as coarsest level)."""
    return lvl if lvl is not None else 0

def _first_var(ds, candidates):
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _read_grd_bbox(path):
    """
    Read any NetCDF/GRD bathymetry file and return (x, y, z, x0, x1, y0, y1).
    Tries multiple variable naming conventions (GMT, CF, GDAL, curvilinear).
    """
    with Dataset(path) as ds:
        xname = _first_var(ds, _X_NAMES)
        yname = _first_var(ds, _Y_NAMES)
        zname = _first_var(ds, _Z_NAMES)

        if xname is None or yname is None or zname is None:
            raise ValueError(
                f'Cannot identify x/y/z variables.\n'
                f'  Available: {list(ds.variables.keys())}\n'
                f'  Expected x-like: {_X_NAMES}\n'
                f'  Expected z-like: {_Z_NAMES}'
            )

        xv = np.array(ds.variables[xname][:])
        yv = np.array(ds.variables[yname][:])
        zv = np.array(ds.variables[zname][:], dtype=float)

        if xv.ndim == 2: xv = xv[0, :]
        if yv.ndim == 2: yv = yv[:, 0]
        if zv.ndim == 3: zv = zv[0]
        if zv.ndim != 2:
            raise ValueError(f'z variable has unexpected shape {zv.shape}')

    return xv, yv, zv, float(xv[0]), float(xv[-1]), float(yv[0]), float(yv[-1])

# ── Colour palette per level ──────────────────────────────────────────────────
LEVEL_COLORS = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9',
                '#8e44ad', '#16a085', '#c0392b', '#f39c12']

def _plot_domain(_=None):
    with out_map:
        clear_output(wait=True)
        parfile = state.get('parfile')
        if not parfile:
            print('⚠  Run Steps 1-2 first and select a parfile.')
            return

        all_grids = _parse_grid_files(parfile)   # (name, path, level_or_None)
        if not all_grids:
            print(f'⚠  No bathymetry files (.grd/.nc/.nc4) found in {os.path.basename(parfile)}')
            return

        # ── Report ────────────────────────────────────────────────────────────
        print(f'Grid files found in parfile: {len(all_grids)}')
        for name, path, lvl in all_grids:
            exists = '✓' if os.path.isfile(path) else '✗  MISSING'
            lvl_eff = _effective_level(lvl)
            lvl_str = f'L{lvl_eff}' + ('' if lvl is not None else '*')
            ext = os.path.splitext(name)[1].upper() or '.GRD'
            print(f'  [{lvl_str}] {ext:6s}  {exists}  {path}')

        # Keep only existing files
        available = [(name, path, lvl) for name, path, lvl in all_grids
                     if os.path.isfile(path)]
        if not available:
            print('\n⚠  No bathymetry grid files accessible from this machine.')
            return

        # Group by effective level
        by_level = {}
        for name, path, lvl in available:
            by_level.setdefault(_effective_level(lvl), []).append((name, path))

        sorted_levels = sorted(by_level.keys())
        print(f'\n  Nesting levels found: {sorted_levels}')
        for lvl in sorted_levels:
            names = ', '.join(n for n, _ in by_level[lvl])
            print(f'  L{lvl}: {len(by_level[lvl])} grid(s) — {names}')

        # ── Overview map ──────────────────────────────────────────────────────
        coarsest_level = sorted_levels[0]
        coarsest_name, coarsest_path = by_level[coarsest_level][0]
        try:
            cx, cy, cz, cx0, cx1, cy0, cy1 = _read_grd_bbox(coarsest_path)
        except Exception as e:
            print(f'\n✗  Cannot read coarsest grid: {e}')
            return

        fig_ov, ax_ov = plt.subplots(figsize=(10, 7))
        vmin = max(np.nanmin(cz), -6000)
        vmax = min(np.nanmax(cz),   500)
        pcm = ax_ov.pcolormesh(cx, cy, cz, cmap='terrain', shading='auto',
                               vmin=vmin, vmax=vmax, zorder=1)
        plt.colorbar(pcm, ax=ax_ov, label='m  (neg = ocean)', shrink=0.85)
        ax_ov.contour(cx, cy, cz, levels=[0], colors='black',
                      linewidths=0.6, zorder=2)

        legend_patches = [
            mpatches.Patch(color=LEVEL_COLORS[coarsest_level % len(LEVEL_COLORS)],
                           label=f'L{coarsest_level} (background)')
        ]

        for lvl in sorted_levels[1:]:
            col = LEVEL_COLORS[lvl % len(LEVEL_COLORS)]
            # Offset (in degrees) so labels of different sub-grids within
            # the same level don't stack on top of each other.
            # Each level gets its label in the TOP-LEFT corner of its box,
            # inset by a small fraction of the box size.
            for sub_idx, (name, path) in enumerate(by_level[lvl]):
                try:
                    _, _, _, x0, x1, y0, y1 = _read_grd_bbox(path)
                    dx = x1 - x0;  dy = y1 - y0
                    rect = mpatches.Rectangle(
                        (x0, y0), dx, dy,
                        linewidth=2, edgecolor=col, facecolor=col,
                        alpha=0.15, zorder=3,
                    )
                    ax_ov.add_patch(rect)
                    ax_ov.plot([x0, x1, x1, x0, x0],
                               [y0, y0, y1, y1, y0],
                               color=col, linewidth=2, zorder=4)
                    # Label OUTSIDE the box: anchor at top-left corner,
                    # offset in points → always outside regardless of scale.
                    # Multiple sub-grids at the same level stack upward.
                    short_name = os.path.splitext(name)[0]
                    if len(short_name) > 20:
                        short_name = short_name[:18] + '…'
                    ax_ov.annotate(
                        f'L{lvl}  {short_name}',
                        xy=(x0, y1),
                        xytext=(3, 3 + sub_idx * 14),
                        textcoords='offset points',
                        ha='left', va='bottom', fontsize=7,
                        color=col, fontweight='bold', zorder=5,
                        bbox=dict(fc='white', ec=col, alpha=0.85,
                                  pad=1.5, boxstyle='round,pad=0.3'),
                    )
                except Exception as err:
                    print(f'  ⚠  Could not read {name}: {err}')
            legend_patches.append(
                mpatches.Patch(color=col, label=f'L{lvl} ({len(by_level[lvl])} grid(s))')
            )

        ax_ov.legend(handles=legend_patches, loc='upper right', fontsize=8)
        ax_ov.set_xlabel('Lon (°E)'); ax_ov.set_ylabel('Lat (°N)')
        ax_ov.set_title(
            f'Domain overview — {os.path.basename(parfile)}\n'
            f'{len(available)} grids · {len(sorted_levels)} nesting levels',
            fontsize=11, fontweight='bold',
        )
        ax_ov.grid(True, linestyle='--', alpha=0.3, zorder=0)
        plt.tight_layout()
        if chk_save.value:
            _stem = os.path.splitext(os.path.basename(parfile))[0]
            _out  = state.get('sim_dir') or os.path.dirname(parfile)
            _p    = os.path.join(_out, f'{_stem}_overview.png')
            fig_ov.savefig(_p, dpi=150, bbox_inches='tight')
            print(f'  💾 Saved: {_p}')
        plt.show()

        # ── Individual subplots ───────────────────────────────────────────────
        n = len(available)
        ncols = min(n, 3)
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6 * ncols, 5 * nrows),
                                 squeeze=False)
        ax_flat = [axes[r][c] for r in range(nrows) for c in range(ncols)]
        for ax in ax_flat[n:]:
            ax.set_visible(False)

        for idx, (name, path, lvl) in enumerate(available):
            ax = ax_flat[idx]
            lvl_eff = _effective_level(lvl)
            col = LEVEL_COLORS[lvl_eff % len(LEVEL_COLORS)]
            ext = os.path.splitext(name)[1].upper() or '.GRD'
            try:
                x, y, z, *_ = _read_grd_bbox(path)
                vm0 = max(np.nanmin(z), -6000)
                vm1 = min(np.nanmax(z),   500)
                ax.pcolormesh(x, y, z, cmap='terrain', shading='auto',
                              vmin=vm0, vmax=vm1)
                ax.contour(x, y, z, levels=[0], colors='black', linewidths=0.8)
                dx_m = abs(float(x[1] - x[0])) * 111320
                ax.set_title(
                    f'L{lvl_eff} [{ext}] · {os.path.splitext(name)[0]}\n'
                    f'{len(x)}×{len(y)} · Δx≈{dx_m:.0f} m',
                    fontsize=9, color=col, fontweight='bold',
                )
                for spine in ax.spines.values():
                    spine.set_edgecolor(col)
                    spine.set_linewidth(2)
            except Exception as e:
                ax.set_title(f'L{lvl_eff} — {name}', fontsize=8)
                ax.text(0.5, 0.5, str(e), transform=ax.transAxes,
                        ha='center', va='center', fontsize=7,
                        color='red', wrap=True)
            ax.set_xlabel('Lon (°E)', fontsize=8)
            ax.set_ylabel('Lat (°N)', fontsize=8)
            ax.grid(True, linestyle='--', alpha=0.3)

        plt.suptitle(f'Individual grids — {os.path.basename(parfile)}',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        if chk_save.value:
            _stem = os.path.splitext(os.path.basename(parfile))[0]
            _out  = state.get('sim_dir') or os.path.dirname(parfile)
            _pp   = os.path.join(_out, f'{_stem}_grids_panel.png')
            fig.savefig(_pp, dpi=150, bbox_inches='tight')
            print(f'  💾 Saved panel: {_pp}')
            for _nm, _pt, _lv in available:
                _le  = _effective_level(_lv)
                _col = LEVEL_COLORS[_le % len(LEVEL_COLORS)]
                try:
                    _x, _y, _z, *_ = _read_grd_bbox(_pt)
                    _fig_s, _ax_s = plt.subplots(figsize=(7, 6))
                    _vm0 = max(np.nanmin(_z), -6000)
                    _vm1 = min(np.nanmax(_z),   500)
                    _pcm = _ax_s.pcolormesh(_x, _y, _z, cmap='terrain',
                                            shading='auto', vmin=_vm0, vmax=_vm1)
                    _ax_s.contour(_x, _y, _z, levels=[0], colors='black',
                                  linewidths=0.8)
                    _dxm    = abs(float(_x[1] - _x[0])) * 111320
                    _stem_g = os.path.splitext(_nm)[0]
                    _ttl = f'L{_le}  {_stem_g}\n{len(_x)}×{len(_y)} · Δx≈{_dxm:.0f} m'
                    _ax_s.set_title(
                        _ttl,
                        fontsize=10, color=_col, fontweight='bold',
                    )
                    for _sp in _ax_s.spines.values():
                        _sp.set_edgecolor(_col); _sp.set_linewidth(2)
                    _ax_s.set_xlabel('Lon (°E)'); _ax_s.set_ylabel('Lat (°N)')
                    _ax_s.grid(True, linestyle='--', alpha=0.3)
                    plt.colorbar(_pcm, ax=_ax_s, label='m  (neg = ocean)', shrink=0.85)
                    plt.tight_layout()
                    _pg = os.path.join(_out, f'{_stem}_L{_le}_{_stem_g}.png')
                    _fig_s.savefig(_pg, dpi=150, bbox_inches='tight')
                    plt.close(_fig_s)
                    print(f'  💾 Saved: {_pg}')
                except Exception as _e:
                    print(f'  ⚠  Could not save {_nm}: {_e}')
        plt.show()

chk_save = widgets.Checkbox(
    value=False,
    description='Save figures as PNG',
    indent=False,
    layout=widgets.Layout(width='220px'),
)
btn_plot = widgets.Button(description='Plot domain', button_style='success', icon='map')
btn_plot.on_click(_plot_domain)
display(widgets.HBox([btn_plot, chk_save]), out_map)

---
## Step 4 — Configure GPUs and run load balancing

Select the number of GPUs, then click **Run load balancing**.  
This must complete successfully before launching the simulation.

In [ ]:
out_lb = widgets.Output()

gpu_slider = widgets.IntSlider(
    value=1, min=1, max=8, step=1,
    description='GPUs:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)
gpu_label = widgets.Label(value='1 GPU selected')

def _update_gpu_label(change):
    n = change['new']
    state['n_gpus'] = n
    gpu_label.value = f'{n} GPU{"s" if n > 1 else ""} selected'

gpu_slider.observe(_update_gpu_label, names='value')
state['n_gpus'] = gpu_slider.value

btn_lb = widgets.Button(
    description='Run load balancing',
    button_style='warning',
    icon='cogs',
    layout=widgets.Layout(width='220px'),
)
lb_status = widgets.HTML(value='')

def _run_lb(_):
    with out_lb:
        clear_output(wait=True)
        d = state['sim_dir']
        pf = state['parfile']
        n = state['n_gpus']

        if not d or not pf:
            print('⚠  Complete Steps 1-2 first.')
            return
        if not os.path.isfile(LB_EXE):
            print(f'✗  LB executable not found: {LB_EXE}')
            print('   (Are you running on the compute server?)')
            return

        pf_name = os.path.basename(pf)
        lb_status.value = '<b style="color:orange">⏳ Running load balancing…</b>'
        print(f'▶  {LB_EXE} {pf_name} 1 {n}')
        print(f'   cwd: {d}\n')

        script = (
            _make_env_preamble() +
            f'cd {d}\n'
            f'{LB_EXE} {pf_name} 1 {n}\n'
        )
        result = subprocess.run(
            ['conda', 'run', '--no-capture-output', '-n', CONDA_ENV, '/bin/bash', '-c', script],
            cwd=d, capture_output=True, text=True,
        )
        print(result.stdout)
        if result.returncode == 0:
            lb_status.value = f'<b style="color:green">✓ Load balancing OK  ({n} GPU{"s" if n>1 else ""})</b>'
            print('✓  Done — ready to launch simulation.')
        else:
            lb_status.value = '<b style="color:red">✗ Load balancing FAILED</b>'
            print('✗  STDERR:')
            print(result.stderr)

btn_lb.on_click(_run_lb)
display(
    widgets.HBox([gpu_slider, gpu_label]),
    widgets.HBox([btn_lb, lb_status]),
    out_lb,
)

---
## Step 5 — Launch simulation

Select the number of GPUs to use, then click **🚀 Launch TsunamiHySEA**.  
The simulation is launched with `mpirun -np <N> TsunamiHySEA parfile` inside `conda run -n <CONDA_ENV>` (set in the configuration cell).  
Output is written to `hysea_run.log` in the simulation directory.

> **Note:** the number of GPUs here must match the value used in Step 4 (load balancing).

In [ ]:
out_launch = widgets.Output()

# ── GPU selector ──────────────────────────────────────────────────────────────
launch_gpu_slider = widgets.IntSlider(
    value=1, min=1, max=8, step=1,
    description='GPUs:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)
launch_gpu_label = widgets.Label(value=f'{MPIRUN_EXE} -np 1 TsunamiHySEA')

def _update_launch_gpu_label(change):
    n = change['new']
    state['n_gpus_launch'] = n
    launch_gpu_label.value = f'{MPIRUN_EXE} -np {n} TsunamiHySEA'

launch_gpu_slider.observe(_update_launch_gpu_label, names='value')
state['n_gpus_launch'] = launch_gpu_slider.value

# ── Consistency check between LB and launch GPUs ──────────────────────────────
lb_warn = widgets.HTML(value='')

def _check_gpu_consistency(change):
    nl = state.get('n_gpus', 1)
    nr = change['new']
    if nl != nr:
        lb_warn.value = (
            f'<b style="color:orange">⚠ GPU mismatch: load balancing used {nl} GPU(s), '
            f'launch set to {nr}. They should match.</b>'
        )
    else:
        lb_warn.value = ''

launch_gpu_slider.observe(_check_gpu_consistency, names='value')

# ── Launch button ──────────────────────────────────────────────────────────────
btn_launch = widgets.Button(
    description='🚀  Launch TsunamiHySEA',
    button_style='danger',
    layout=widgets.Layout(width='260px', height='40px'),
)
launch_status = widgets.HTML(value='')

def _launch(_):
    with out_launch:
        clear_output(wait=True)
        d  = state['sim_dir']
        pf = state['parfile']
        n  = state['n_gpus_launch']

        if not d or not pf:
            print('⚠  Complete Steps 1-2 first.')
            return
        if not os.path.isfile(HYSEA_EXE):
            print(f'✗  TsunamiHySEA not found: {HYSEA_EXE}')
            print('   (Are you running on the compute server?)')
            return

        pf_name  = os.path.basename(pf)
        log_file = os.path.join(d, 'hysea_run.log')
        state['log_file'] = log_file

        cmd = f'{MPIRUN_EXE} -np {n} {HYSEA_EXE} {pf_name}'
        script = (
            _make_env_preamble() +
            f'cd {d}\n'
            f'{cmd}\n'
        )
        log_fh = open(log_file, 'w')
        proc = subprocess.Popen(
            ['conda', 'run', '--no-capture-output', '-n', CONDA_ENV, '/bin/bash', '-c', script],
            cwd=d, stdout=log_fh, stderr=subprocess.STDOUT,
            start_new_session=True,   # new session → own PGID → killpg kills the whole tree
        )
        state['proc'] = proc

        launch_status.value = f'<b style="color:green">✓ Running  (PID {proc.pid})</b>'
        print(f'✓  TsunamiHySEA started  (PID {proc.pid})')
        print(f'   command : {cmd}')
        print(f'   parfile : {pf_name}')
        print(f'   log     : {log_file}')
        print('\nScroll down to Step 6 to monitor progress.')


# ── Stop button ───────────────────────────────────────────────────────────────
btn_stop = widgets.Button(
    description='⏹  Stop simulation',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='40px'),
    disabled=True,
)
stop_status = widgets.HTML(value='')

def _stop(_):
    """Kill the whole simulation tree.

    Three complementary strategies, applied together:
    1. psutil.children(recursive=True) — parent-child tree walk
    2. SID filter — catches processes that changed PGID but kept the session
    3. Name filter — catches workers that called setsid() (new SID, new PGID)
       using pre_launch_pids to avoid killing unrelated TsunamiHySEA runs
    After collecting all candidates: SIGTERM → 3 s wait → SIGKILL survivors.
    """
    import time as _time
    proc = state.get('proc')
    if proc is None:
        stop_status.value = '<b style="color:orange">⚠ No simulation running.</b>'
        return
    if proc.poll() is not None:
        stop_status.value = '<b style="color:grey">— Simulation already finished —</b>'
        state['proc'] = None
        btn_stop.disabled = True
        btn_launch.disabled = False
        return

    stop_status.value = '<b style="color:orange">⏳ Stopping...</b>'

    # ── Get the Session ID of our job ─────────────────────────────────────────
    try:
        job_sid = os.getsid(proc.pid)
    except ProcessLookupError:
        state['proc'] = None
        btn_stop.disabled = True
        btn_launch.disabled = False
        stop_status.value = '<b style="color:grey">— Process already gone —</b>'
        return


    # ── Collect all targets (three strategies) ────────────────────────────────
    job_procs = []
    seen_pids = {os.getpid()}   # never kill ourselves

    if HAS_PSUTIL:
        try:
            import psutil as _psutil

            # Strategy 1: parent-child tree walk
            try:
                root = _psutil.Process(proc.pid)
                for p in root.children(recursive=True):
                    if p.pid not in seen_pids:
                        job_procs.append(p)
                        seen_pids.add(p.pid)
                if proc.pid not in seen_pids:
                    job_procs.append(root)
                    seen_pids.add(proc.pid)
            except Exception:
                pass

            # Strategy 2: same Session ID (catches PGID-escaped processes)
            for p in _psutil.process_iter(['pid']):
                if p.pid in seen_pids:
                    continue
                try:
                    if os.getsid(p.pid) == job_sid:
                        job_procs.append(p)
                        seen_pids.add(p.pid)
                except (ProcessLookupError, PermissionError, OSError):
                    pass

            # Strategy 3: TsunamiHySEA by name, owned by the current user
            # (catches workers that called setsid() → new SID, new PGID,
            #  invisible to strategies 1 and 2; username filter avoids
            #  killing other users' simulations on shared HPC nodes)
            import getpass as _getpass
            _me = _getpass.getuser()
            for p in _psutil.process_iter(['pid', 'name', 'username']):
                if p.pid in seen_pids:
                    continue
                try:
                    if ('TsunamiHySEA' in (p.info.get('name') or '') and
                            p.info.get('username') == _me):
                        job_procs.append(p)
                        seen_pids.add(p.pid)
                except Exception:
                    pass

        except Exception:
            pass

    # Fallback if psutil unavailable: killpg on root PGID
    if not job_procs:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            pass

    # ── Step 1: SIGTERM ───────────────────────────────────────────────────────
    killed_pids = []
    for p in job_procs:
        try:
            p.send_signal(signal.SIGTERM)
            killed_pids.append(p.pid)
        except Exception:
            pass

    # ── Step 2: wait up to 3 s ────────────────────────────────────────────────
    deadline = _time.monotonic() + 3.0
    while _time.monotonic() < deadline:
        if proc.poll() is not None:
            break
        _time.sleep(0.2)

    killed_with = 'SIGTERM'
    if proc.poll() is None:
        # ── Step 3: SIGKILL any survivor ──────────────────────────────────────
        for p in job_procs:
            try:
                p.send_signal(signal.SIGKILL)
            except Exception:
                pass
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            pass
        killed_with = 'SIGKILL'

    state['proc'] = None
    btn_stop.disabled = True
    btn_launch.disabled = False
    launch_status.value = '<b style="color:grey">— Stopped by user —</b>'
    n = len(killed_pids)
    stop_status.value = (
        f'<b style="color:green">✓ {n} process(es) terminated ({killed_with}): '
        f'{killed_pids}</b>'
    )
    with out_launch:
        from IPython.display import clear_output
        clear_output(wait=True)
        print(f'✓ Simulation stopped. {n} process(es) killed with {killed_with}.')
        print(f'  PIDs: {killed_pids}')

btn_stop.on_click(_stop)

# Enable/disable stop button in sync with launch
_orig_launch = _launch
def _launch_with_stop(b):
    _orig_launch(b)
    btn_stop.disabled  = False
    btn_launch.disabled = True
# _launch is wrapped (not registered directly) so a single click runs it once,
# then toggles the Stop/Launch buttons.
btn_launch.on_click(_launch_with_stop)

display(
    widgets.HBox([launch_gpu_slider, launch_gpu_label]),
    lb_warn,
    widgets.HBox([btn_launch, launch_status, btn_stop, stop_status]),
    out_launch,
)

---
## Step 6 — Monitor simulation

Click **Refresh log** to read the latest lines from `hysea_run.log`.  
Enable **Auto-refresh** to update every 10 seconds automatically.

In [ ]:
import re as _re

# ── Parfile: extract total simulation time ────────────────────────────────────
# Format 1 (commented):  14401.0   # Simulation time (sec)
# Format 2 (positional): the number following the 4 border-condition lines (1 or -1)

_RE_SIMTIME_CMT = _re.compile(
    r'^\s*([\d.eE+\-]+)\s*#.*[Ss]imulation\s+time', _re.MULTILINE
)
_RE_BORDER = _re.compile(r'^\s*(-?1)\s*(?:#.*)?$')


def _read_parfile_safe(path):
    """Read parfile text with UTF-8 (BOM-tolerant) and latin-1 fallback.
    Returns '' on any error so callers can keep working silently."""
    if not path or not os.path.isfile(path):
        return ''
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            with open(path, encoding=enc) as fh:
                return fh.read()
        except UnicodeDecodeError:
            continue
        except Exception:
            return ''
    return ''


def _parse_sim_time(parfile_path):
    """Return total simulation time (float, seconds) from the parfile, or None."""
    text = _read_parfile_safe(parfile_path)
    if not text:
        return None

    # Strategy 1: explicit comment
    m = _RE_SIMTIME_CMT.search(text)
    if m:
        return float(m.group(1))

    # Strategy 2: positional — find 4 consecutive border-condition lines (1 or -1),
    # then the next non-empty/non-comment line is the simulation time
    lines = text.splitlines()
    consecutive = 0
    for i, ln in enumerate(lines):
        stripped = ln.strip()
        if not stripped or stripped.startswith('#'):
            continue
        if _RE_BORDER.match(ln):
            consecutive += 1
            if consecutive == 4:
                # Find next data line
                for j in range(i + 1, min(i + 5, len(lines))):
                    nxt = lines[j].split('#')[0].strip()
                    if nxt:
                        try:
                            return float(nxt)
                        except ValueError:
                            pass
                break
        else:
            consecutive = 0
    return None

# ── Log: extract current simulation time ─────────────────────────────────────
# Matches: "Iteration 3095, deltaT = 2.32e+00 sec, Time = 7180.4 sec"

_RE_LOG_TIME = _re.compile(
    r'[Ii]teration\s+(\d+).*?[Tt]ime\s*=\s*([\d.eE+\-]+)\s*sec'
)
_RE_LOG_IT_ONLY = _re.compile(r'[Ii]teration\s+(\d+)')

def _extract_progress(log_path, total_time=None):
    """
    Parse log tail and return (pct: float|None, label: str).
    Uses total_time (from parfile) to compute percentage.
    """
    if not log_path or not os.path.isfile(log_path):
        return None, ''
    try:
        with open(log_path, 'rb') as fh:
            fh.seek(0, 2)
            size = fh.tell()
            fh.seek(max(0, size - 16384))
            tail = fh.read().decode('utf-8', errors='replace')
    except Exception:
        return None, ''

    cur_time = None
    cur_iter = None

    for m in _RE_LOG_TIME.finditer(tail):
        cur_iter = int(m.group(1))
        cur_time = float(m.group(2))

    if cur_time is None:
        # Fallback: iteration-only lines
        for m in _RE_LOG_IT_ONLY.finditer(tail):
            cur_iter = int(m.group(1))

    if cur_time is not None and total_time and total_time > 0:
        pct = min(100.0, 100.0 * cur_time / total_time)
        label = (
            f'Iteration {cur_iter} · '
            f'Time {cur_time:.1f} / {total_time:.0f} s · '
            f'<b>{pct:.1f}%</b>'
        )
        return pct, label

    if cur_iter is not None:
        return None, f'Iteration {cur_iter}  (total time unknown)'

    return None, ''

# ── Widgets ───────────────────────────────────────────────────────────────────
progress_bar = widgets.FloatProgress(
    value=0, min=0, max=100,
    description='Progress:',
    bar_style='info',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='65%', height='26px'),
)
progress_label = widgets.HTML(value='<span style="color:#888">—</span>')
proc_status    = widgets.HTML(value='')
simtime_info   = widgets.HTML(value='')

out_monitor = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ccc', padding='8px',
        height='280px', overflow_y='auto',
        width='95%',
    )
)

btn_refresh    = widgets.Button(description='Refresh', button_style='info',
                                icon='refresh', layout=widgets.Layout(width='100px'))
n_lines_slider = widgets.IntSlider(
    value=30, min=10, max=200, step=10,
    description='Lines:',
    layout=widgets.Layout(width='280px'),
)
auto_toggle = widgets.ToggleButton(
    value=False, description='Auto-refresh (10 s)',
    button_style='', icon='clock-o',
)

_auto_thread = [None]
_stop_auto   = [False]

# ── Read log tail ─────────────────────────────────────────────────────────────
def _read_log(n=30):
    log = state.get('log_file')
    if not log:
        d = state.get('sim_dir') or '.'
        candidates = (
            glob.glob(os.path.join(d, 'hysea_run.log')) +
            glob.glob(os.path.join(d, 'simulation.out')) +
            sorted(glob.glob(os.path.join(d, 'slurm-*.out')), reverse=True)
        )
        if not candidates:
            return None, '⚠  No log file found yet.'
        log = candidates[0]
    try:
        # UTF-8 with replacement — HySEA logs may contain non-ASCII chars
        with open(log, encoding='utf-8', errors='replace') as fh:
            lines = fh.readlines()
        tail = ''.join(lines[-n:])
        return log, tail if tail else '(log is empty)'
    except Exception as e:
        return log, f'Error reading log: {e}'

# ── Refresh callback ──────────────────────────────────────────────────────────
def _refresh(_=None):
    log_path   = state.get('log_file')
    parfile    = state.get('parfile')
    total_time = _parse_sim_time(parfile)

    # Show total time info once
    if total_time:
        simtime_info.value = (
            f'<span style="color:#555">Total simulation time: '
            f'<b>{total_time:.0f} s</b> ({total_time/3600:.2f} h)</span>'
        )
    else:
        simtime_info.value = (
            '<span style="color:#e67e22">⚠ Could not read simulation time from parfile</span>'
        )

    # Progress bar
    pct, plabel = _extract_progress(log_path, total_time)
    if pct is not None:
        progress_bar.value = pct
        progress_label.value = plabel
        progress_bar.bar_style = 'success' if pct >= 100 else 'info'
    else:
        progress_bar.value   = 0
        progress_label.value = (
            f'<span style="color:#888">{plabel or "waiting for first iteration…"}</span>'
        )

    # Process status
    proc = state.get('proc')
    if proc:
        rc = proc.poll()
        if rc is None:
            proc_status.value = f'<b style="color:green">● Running  (PID {proc.pid})</b>'
        elif rc == 0:
            proc_status.value      = '<b style="color:#2980b9">✓ Finished (exit 0)</b>'
            progress_bar.value     = 100
            progress_bar.bar_style = 'success'
            progress_label.value   = '<b>100% — done</b>'
        else:
            proc_status.value      = f'<b style="color:red">✗ Exited with code {rc}</b>'
            progress_bar.bar_style = 'danger'

    # Log tail — write via the widget's own methods (NOT 'with out_monitor:').
    # The Output context manager swaps the global sys.stdout, which leaks into
    # other cells when _refresh runs from the background auto-refresh thread.
    log, content = _read_log(n_lines_slider.value)
    header = (f'[ {log} ]  —  last {n_lines_slider.value} lines\n' + '─' * 60 + '\n') if log else ''
    out_monitor.clear_output(wait=True)
    out_monitor.append_stdout(header + (content or '') + '\n')

# ── Auto-refresh loop ─────────────────────────────────────────────────────────
def _auto_refresh_loop():
    while not _stop_auto[0]:
        try:
            _refresh()
        except Exception as _e:
            # Never let a transient error kill the loop or spam tracebacks
            # (e.g. log/dir not ready yet). Surface it quietly in the monitor.
            out_monitor.clear_output(wait=True)
            out_monitor.append_stdout(f'(auto-refresh waiting — {type(_e).__name__}: {_e})\n')
        for _ in range(100):
            if _stop_auto[0]: break
            time.sleep(0.1)

def _toggle_auto(change):
    if change['new']:
        _stop_auto[0] = False
        t = threading.Thread(target=_auto_refresh_loop, daemon=True)
        _auto_thread[0] = t
        t.start()
        auto_toggle.description = 'Stop auto-refresh'
        auto_toggle.button_style = 'warning'
    else:
        _stop_auto[0] = True
        auto_toggle.description = 'Auto-refresh (10 s)'
        auto_toggle.button_style = ''

btn_refresh.on_click(_refresh)
auto_toggle.observe(_toggle_auto, names='value')

display(
    widgets.HBox([btn_refresh, n_lines_slider, auto_toggle, proc_status]),
    simtime_info,
    widgets.HBox([progress_bar, progress_label],
                 layout=widgets.Layout(align_items='center', margin='4px 0')),
    out_monitor,
)